In [ ]:
import numpy as np
import time
from pynq import Overlay, allocate

BASE = '/home/xilinx/jupyter_notebooks/fpga-prosthetic-poc/'

print('오버레이 로드 중...')
overlay = Overlay(BASE + 'svm_overlay.bit')
print('완료')
print(overlay.ip_dict.keys())

In [ ]:
# HLS IP 접근
svm_ip = overlay.svm_inference_0

# 파라미터 로드
p = np.load(BASE + 'svm_params.npz')
scaler_mean = p['scaler_mean']
scaler_std  = p['scaler_std']

print('IP 레지스터 맵:')
print(svm_ip.register_map)

In [ ]:
def fpga_predict(x_raw):
    # 입력 정규화
    x_scaled = ((x_raw - scaler_mean) / scaler_std).astype(np.float32)

    # AXI Lite로 입력 전송
    for i, val in enumerate(x_scaled):
        svm_ip.write(0x10 + i * 4, int(np.float32(val).view(np.uint32)))

    # 실행 시작
    svm_ip.write(0x00, 1)

    # 완료 대기
    while not (svm_ip.read(0x00) & 0x2):
        pass

    # 결과 읽기
    pred = svm_ip.read(0x10 + 96 * 4)
    return pred

# 테스트
X_test = np.random.randn(96).astype(np.float32)
print('예측 결과:', fpga_predict(X_test))

In [ ]:
# 워밍업
for _ in range(10):
    fpga_predict(X_test)

# 지연시간 측정 (100회)
latencies = []
for _ in range(100):
    start = time.perf_counter()
    fpga_predict(X_test)
    end = time.perf_counter()
    latencies.append((end - start) * 1000)

print('[FPGA 가속 결과]')
print(f'  평균 지연시간: {np.mean(latencies):.3f} ms')
print(f'  표준편차:     {np.std(latencies):.3f} ms')
print(f'  최소:         {np.min(latencies):.3f} ms')
print(f'  최대:         {np.max(latencies):.3f} ms')
print()
arm_baseline = 24.580
print(f'ARM baseline:  {arm_baseline:.3f} ms')
print(f'속도 향상:     {arm_baseline / np.mean(latencies):.1f}x')